In [ ]:
!rm -rf '/content/DIS_Hughen'

In [ ]:
!git clone https://github.com/NU-Academics/DIS_Hughen.git

In [11]:
import numpy as np
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import  accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [3]:
# df = pd.read_csv("undersampled_CIC2019_dataset.csv")
df = pd.read_csv("/content/DIS_Hughen/undersampled_CIC2019_dataset.csv")

In [4]:
df.shape

(4685611, 90)

In [6]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["label"])
X = df.drop(columns=["label"], errors="ignore").select_dtypes(include=[np.number])
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(0, inplace=True)
y = df["label"]
label_mapping = dict(zip(le.classes_, range(len(le.classes_))))
print(label_mapping)

{'BENIGN': 0, 'DrDoS_DNS': 1, 'DrDoS_LDAP': 2, 'DrDoS_MSSQL': 3, 'DrDoS_NTP': 4, 'DrDoS_NetBIOS': 5, 'DrDoS_SNMP': 6, 'DrDoS_SSDP': 7, 'DrDoS_UDP': 8, 'LDAP': 9, 'MSSQL': 10, 'NetBIOS': 11, 'Portmap': 12, 'Syn': 13, 'TFTP': 14, 'UDP': 15, 'UDP-lag': 16, 'UDPLag': 17, 'WebDDoS': 18}


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)

In [10]:
class CICDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = CICDataset(X_train, y_train)
train_loader = DataLoader(train_dataset,
                          batch_size=8192,
                          shuffle=True,
                          pin_memory=True,
                          num_workers=4)

In [14]:
import torch.nn as nn
import torch.nn.functional as F

class DNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.dropout1 = nn.Dropout(0.3)

        self.fc2 = nn.Linear(256, 128)
        self.dropout2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(128, 64)

        self.output = nn.Linear(64, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        features = F.relu(self.fc3(x))
        logits = self.output(features)
        return logits, features

In [15]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)

In [16]:
model = DNN(input_dim=X_train.shape[1],
            num_classes=len(np.unique(y_train))).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(15):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits, _ = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

RuntimeError: DataLoader worker (pid(s) 38276, 10500, 2104, 33840) exited unexpectedly

In [ ]:
def extract_features(model, X, batch_size=8192):
    model.eval()
    dataset = torch.tensor(X.values, dtype=torch.float32)
    loader = DataLoader(dataset, batch_size=batch_size)

    features_list = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            _, features = model(batch)
            features_list.append(features.cpu().numpy())

    return np.vstack(features_list)

Z_train = extract_features(model, X_train)
Z_test = extract_features(model, X_test)

In [ ]:
X_train_hybrid = np.hstack([X_train.values, Z_train])
X_test_hybrid = np.hstack([X_test.values, Z_test])

In [5]:
model_classifier = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    device='cuda'
)

In [11]:
y_pred = model_classifier.predict(Z_test)
y_prob = model_classifier.predict_proba(Z_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Weighted F1:", f1_score(y_test, y_pred, average='weighted'))
print("Macro F1:", f1_score(y_test, y_pred, average='macro'))
print("ROC-AUC:", roc_auc_score(y_test, y_prob, multi_class="ovr", average="weighted"))

Accuracy: 0.7630460462500653
Weighted F1: 0.7370615293819677
Macro F1: 0.625859427843666
ROC-AUC: 0.9733469812240148


In [12]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, target_names=le.classes_))

               precision    recall  f1-score   support

       BENIGN       0.98      0.99      0.99     10655
    DrDoS_DNS       0.68      0.84      0.75    178752
   DrDoS_LDAP       0.59      0.13      0.21     58000
  DrDoS_MSSQL       0.70      0.32      0.44     30000
    DrDoS_NTP       1.00      1.00      1.00    183522
DrDoS_NetBIOS       0.80      0.79      0.80     13000
   DrDoS_SNMP       0.52      0.52      0.52     83000
   DrDoS_SSDP       0.65      0.06      0.11     21330
    DrDoS_UDP       0.53      0.04      0.08     21000
         LDAP       0.49      0.49      0.49     58000
        MSSQL       0.57      0.86      0.68     38000
      NetBIOS       0.77      0.86      0.81     11011
      Portmap       0.70      0.44      0.54      1122
          Syn       0.99      1.00      0.99      8401
         TFTP       1.00      1.00      1.00    190059
          UDP       0.42      0.97      0.59     30000
      UDP-lag       1.00      0.06      0.12      1000
       UD